# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR\(^2\) dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id` as specified in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# dataset.metadata is a mlcroissant Metadata object.
metadata_json = dataset.metadata.to_json()
print(f"Name: {metadata_json.get('name')}")
print(f"Description: {metadata_json.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

First, let's list all Record Set `@id`s, along with the fields present in each Record Set. This information is crucial for data extraction and further analysis.

In [ ]:
# List all available record sets and their fields, referencing by @id
all_record_sets = dataset.metadata.record_sets
print("Available record sets and their fields (referenced by @id):\n")
for rs in all_record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print("  Fields (by @id):")
    for field in rs.fields:
        column_ids = [col.id for col in field.columns]
        print(f"    Field @id: {field.id}, Name: {field.name}, Columns @id: {column_ids}")
    print('-'*60)
if not all_record_sets:
    print("There are no explicit record sets found in the Croissant schema. Please verify schema details if expected record sets are missing.")

Let's display a few records from a chosen Record Set using its `@id`. This will help us understand the structure and sample data.

> **Note:** Replace `<record_set_id>` with the actual `@id` for the RecordSet you'd like to inspect, as listed above.

In [ ]:
# Choose a RecordSet @id from the list above.
# As an example, we will use the first available record set.
if all_record_sets:
    example_record_set_id = all_record_sets[0].id
    print(f"Showing a few records from RecordSet: {example_record_set_id}\n")
    try:
        for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
            pprint.pprint(record)
            if i >= 2:
                break  # Show only a few samples
    except Exception as e:
        print(f"Could not extract records for RecordSet {example_record_set_id}: {e}")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from one or more Record Sets into pandas DataFrames for analysis. Use the Record Set and Field `@id`s from the overview above.

We'll dynamically extract data for all discovered RecordSets and store them in a dictionary for convenience, keyed by their `@id`.

In [ ]:
# Extract data from all available RecordSets
dataframes = {}
record_set_ids = [rs.id for rs in all_record_sets]
print("Extracting data from record sets:", record_set_ids)
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded dataframe for RecordSet {record_set_id}. Shape: {df.shape}")
    except Exception as e:
        print(f"Could not load data for {record_set_id}: {e}")

# Display column names for one record set (the first one, as an example)
if record_set_ids:
    default_rs_id = record_set_ids[0]
    if default_rs_id in dataframes:
        print(f"Columns in {default_rs_id}: {dataframes[default_rs_id].columns.tolist()}")
        dataframes[default_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field for filtering and normalization and group by a categorical attribute if available.

> **Note**: Make sure to update `numeric_field_id` and `group_field_id` with relevant values from section 2 if you have a specific field you want to analyze.

In [ ]:
# Setup: Pick a RecordSet and a field for analysis
# For demonstration, we use the first loaded recordset and look for numeric columns
import numpy as np

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Identify potential numeric fields
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields in {record_set_id}: {numeric_fields}")

    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use the first numeric column
        threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].isnull().all() else 0  # Use Q3 as threshold

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in {record_set_id} with '{numeric_field_id}' > {threshold:.2f} (Q3): {len(filtered_df)} records\n")
        print(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field if available
        group_fields = df.select_dtypes(include=[object]).columns.tolist()
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id, dropna=False).mean(numeric_only=True)
            print(f"\nGrouped (mean) data by '{group_field_id}':\n", grouped_df.head())
    else:
        print(f"No numeric fields found in {record_set_id} for EDA.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll produce a histogram and a boxplot for the chosen numeric field, and a scatter plot if a suitable pair of numeric columns exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_fields:
    df_plot = dataframes[record_set_id]
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df_plot[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in record set {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df_plot[numeric_field_id].dropna())
    plt.title(f"Boxplot of '{numeric_field_id}'")
    plt.show()

    # Scatter plot if two numeric fields
    if len(numeric_fields) > 1:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=df_plot[numeric_fields[0]], y=df_plot[numeric_fields[1]])
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.title(f"Scatter plot of {numeric_fields[0]} vs {numeric_fields[1]}")
        plt.show()
else:
    print("Not enough numeric fields for visualization.")

## 6. Conclusion
In this notebook, we have:
- Loaded dataset metadata and records using `mlcroissant`, referencing all entities by their `@id`
- Explored available `RecordSet`s and their fields
- Loaded data into pandas DataFrames
- Performed exploratory data analysis with simple numeric filtering, normalization, and grouping
- Visualized important numeric fields

**You can adapt this workflow to work with other fields and record sets as needed by referencing their `@id` values as shown above.**